[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-2b-occams-razor/EMP5027-Practical-Occams-Razor.ipynb)


# Practical: Occam's Razor in Environmental Data Science

**Aim:** this practical is a short, hands-on demonstration of a principle that matters every time you build
a predictive model: start simple, and only add complexity when it actually earns its place. The principle
is often called Occam's Razor, after the medieval philosopher William of Ockham, and in modelling terms it
means that among models which explain the data reasonably well, we prefer the simplest one, unless a more
complex model gives a genuine, reproducible improvement.

We will work with a small synthetic dataset relating rainfall to stream nitrate concentration, a
relationship you will recognise from catchment hydrology and water quality monitoring. The point of using
synthetic data is that we know the true underlying relationship, so we can see exactly when a model is
picking up real signal and when it is just fitting noise.

**You will learn to:**
- Generate a small synthetic environmental dataset (rainfall predicting nitrate concentration).
- Visualise and summarise the relationship between the two variables.
- Fit a simple linear regression and interpret its coefficients directly.
- Fit a more flexible polynomial model and a black box random forest on the same data.
- Compare train versus test performance, and use cross-validation, to detect overfitting.
- Write a short, decision-maker-friendly interpretation of which model you would actually recommend.


## 0) Setup


## Running this in Google Colab

Click the badge above to open this notebook directly in Colab, no local setup required. Everything this
notebook needs is already available on Colab by default.


In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips this step) ---
import sys

# sys.modules is the dictionary of every module Python has already imported in this session.
# Colab pre-imports "google.colab" automatically, so checking for it is a reliable way to tell
# whether this notebook is running on Colab or on your own machine.
if "google.colab" in sys.modules:
    pass
    print("Running in Colab, ready to go.")
else:
    print("Not running in Colab, assuming packages are already installed locally.")


In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Modelling & evaluation
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Make plots a touch larger and sharper, so they're easier to read on screen
plt.rcParams['figure.figsize'] = (6,4)
plt.rcParams['figure.dpi'] = 120


## 1) Create a tiny synthetic dataset

We are going to simulate a simple, mostly linear environmental process. Think of it as a stand-in for a
real catchment, where rainfall washes nitrate from agricultural land into a stream.

- **Input**: rainfall, in millimetres.
- **Output**: nitrate concentration, in mg per litre.
- **True process** (known to us here, but hidden from the models we fit): nitrate is approximately equal
  to 2 plus 0.18 times rainfall, plus some random noise.
- We also add a small non-linear wiggle to that relationship, so that a more flexible model genuinely has
  a little extra structure it could find. The question this practical asks is whether that extra structure
  is worth chasing, or whether a model that goes looking for it ends up fitting the noise instead.

Because the dataset is synthetic and small, you can run everything below quickly on any laptop, and because
we know the true relationship, we can judge afterwards whether each model recovered something real or
learned a coincidence.


In [ ]:
np.random.seed(42)            # fix the random seed so everyone gets the same numbers

n = 120                                  # total number of samples (think of these as 120 rainfall events)
rainfall = np.random.uniform(0, 80, n)   # rainfall in mm, drawn uniformly between 0 and 80
noise    = np.random.normal(0, 2.0, n)   # measurement and process noise, mean 0, sd 2.0 mg/L

# Mostly linear relationship, plus a *tiny* quadratic wiggle
nitrate = 2 + 0.18 * rainfall + noise
nitrate += 0.002 * (rainfall - 40)**2 * 0.1   # small non-linear tweak, largest away from rainfall = 40 mm

# Collect everything into a tidy DataFrame, one row per sample, with clear column names
df = pd.DataFrame({'rainfall_mm': rainfall, 'nitrate_mgL': nitrate})
df.head()   # peek at the first five rows to sanity check the data looks sensible


## 2) Train/Test Split

Before fitting anything, we set aside a portion of the data purely for testing. The model never sees this
test set during training, so its performance there tells us how well the model generalises to new data,
rather than how well it has simply memorised the data it was fit on. This is the basic tool we use to
detect overfitting throughout the rest of the practical.


In [ ]:
X = df[['rainfall_mm']].values    # scikit-learn expects a 2D array of features, hence the double brackets
y = df['nitrate_mgL'].values      # the target variable, as a 1D array

# Split into training data (70%) and test data (30%), keeping the split fixed with random_state
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

len(X_train), len(X_test)   # check the split sizes look right


## 3) Quick Visual Check

Before fitting any model, it pays to just look at the data. A scatter plot already tells us a lot: does the
relationship look roughly linear, is there a lot of scatter around the trend, are there any obvious
outliers. Get into the habit of doing this before modelling, not after.


In [ ]:
# Scatter plot of the training data only, so we look at the same data the model will be fit on
plt.scatter(X_train, y_train, alpha=0.7, label='Train data')
plt.xlabel('Rainfall (mm)')
plt.ylabel('Nitrate (mg/L)')
plt.title('Scatter: Rainfall vs Nitrate (Train)')
plt.legend()
plt.show()


## 4) Simple, Interpretable Model: Linear Regression

This is our Occam's Razor baseline, the simplest model that could plausibly describe this relationship. We
fit a straight line of the form y = m times x plus c, where m is the slope and c is the intercept, and then
inspect:

- The **equation** itself, since with a linear model you can read the slope and intercept straight off and
  explain them in one sentence.
- **R²** (the proportion of variance in nitrate explained by rainfall) and **RMSE** (root mean squared
  error, in the same units as nitrate) on both the **train** and **test** sets.

Comparing train and test performance here gives us a baseline for what "good" looks like on this dataset,
before we see whether extra model complexity improves on it.


In [ ]:
lin = LinearRegression()
lin.fit(X_train, y_train)   # find the slope and intercept that best fit the training data

# Generate predictions on both the training data and the held-out test data
yhat_train_lin = lin.predict(X_train)
yhat_test_lin  = lin.predict(X_test)

# R-squared: proportion of variance in nitrate explained by the model (1.0 is a perfect fit)
r2_train_lin = r2_score(y_train, yhat_train_lin)
r2_test_lin  = r2_score(y_test,  yhat_test_lin)
# RMSE: typical prediction error, in mg/L, so it's directly comparable to the nitrate values themselves
rmse_train_lin = np.sqrt(mean_squared_error(y_train, yhat_train_lin))
rmse_test_lin  = np.sqrt(mean_squared_error(y_test,  yhat_test_lin))

print('=== Linear Regression (simple, interpretable) ===')
print(f'Equation: nitrate = {lin.intercept_:.2f} + {lin.coef_[0]:.3f} * rainfall')
print(f'R² (train): {r2_train_lin:.3f} | RMSE (train): {rmse_train_lin:.3f}')
print(f'R² (test) : {r2_test_lin:.3f}  | RMSE (test) : {rmse_test_lin:.3f}')


In [ ]:
# Visualise the fitted line against the data
xs = np.linspace(X.min(), X.max(), 200).reshape(-1,1)   # 200 evenly spaced rainfall values, for a smooth line
ys_lin = lin.predict(xs)                                 # predicted nitrate at each of those rainfall values

plt.scatter(X_train, y_train, alpha=0.5, label='Train')
plt.scatter(X_test,  y_test,  alpha=0.7, label='Test')
plt.plot(xs, ys_lin, lw=3, label='Linear fit', zorder=5)
plt.xlabel('Rainfall (mm)'); plt.ylabel('Nitrate (mg/L)')
plt.title('Linear Model vs Data'); plt.legend(); plt.show()


## 5) More Complex: Polynomial Regression (degree = 5)

We now let the model fit a much more flexible curve, a fifth degree polynomial, rather than a straight
line. This gives the model far more freedom to bend and twist to match the training points, which can
reduce the training error, but that flexibility is exactly what risks overfitting: a model with enough
freedom can chase the noise in the training data rather than the true underlying signal.

We compare train versus test performance again. If the curve hugs the training points closely but does
noticeably worse on the test data, that gap is the classic signature of overfitting, and it is the reason
we insist on checking test performance rather than trusting training performance alone.


In [ ]:
# Build a pipeline: first expand rainfall into polynomial features (rainfall, rainfall^2, ... rainfall^5),
# then fit an ordinary linear regression on top of those expanded features
poly5 = Pipeline([
    ('poly', PolynomialFeatures(degree=5, include_bias=False)),
    ('lin', LinearRegression())
])
poly5.fit(X_train, y_train)

yhat_train_p5 = poly5.predict(X_train)
yhat_test_p5  = poly5.predict(X_test)

r2_train_p5   = r2_score(y_train, yhat_train_p5)
r2_test_p5    = r2_score(y_test,  yhat_test_p5)
rmse_train_p5 = np.sqrt(mean_squared_error(y_train, yhat_train_p5))
rmse_test_p5  = np.sqrt(mean_squared_error(y_test,  yhat_test_p5))

print('=== Polynomial Regression (degree=5) ===')
print(f'R² (train): {r2_train_p5:.3f} | RMSE (train): {rmse_train_p5:.3f}')
print(f'R² (test) : {r2_test_p5:.3f}  | RMSE (test) : {rmse_test_p5:.3f}')
print('Note: If train performance is much better than test performance, that suggests overfitting.')


In [ ]:
# Visualise both curves together, so we can see directly how much the polynomial deviates from the line
ys_p5 = poly5.predict(xs)

plt.scatter(X_train, y_train, alpha=0.4, label='Train')
plt.scatter(X_test,  y_test,  alpha=0.7, label='Test')
plt.plot(xs, ys_lin, lw=3, label='Linear fit')
plt.plot(xs, ys_p5,  lw=2, label='Polynomial (deg=5)')
plt.xlabel('Rainfall (mm)'); plt.ylabel('Nitrate (mg/L)')
plt.title('Linear vs Polynomial'); plt.legend(); plt.show()


## 6) Black Box Baseline: Random Forest

A random forest is an ensemble of many decision trees, and it can often perform well on this kind of
problem, but it comes at a cost: it is much less interpretable than a single straight line. You cannot read
off a slope and intercept from a random forest and explain it in one sentence to a colleague or a
policymaker.

Here we check whether that extra complexity meaningfully improves test performance. If it does not, the
interpretability cost of the random forest is not worth paying for this dataset.


In [ ]:
# 300 trees, each grown on a bootstrap sample of the training data, with predictions averaged across trees
# min_samples_leaf=3 stops individual trees from growing leaves with fewer than 3 samples, which limits
# how far the model can overfit to individual noisy points
rf = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=3
)
rf.fit(X_train, y_train)

yhat_train_rf = rf.predict(X_train)
yhat_test_rf  = rf.predict(X_test)

r2_train_rf   = r2_score(y_train, yhat_train_rf)
r2_test_rf    = r2_score(y_test,  yhat_test_rf)
rmse_train_rf = np.sqrt(mean_squared_error(y_train, yhat_train_rf))
rmse_test_rf  = np.sqrt(mean_squared_error(y_test,  yhat_test_rf))

print('=== Random Forest (black box) ===')
print(f'R² (train): {r2_train_rf:.3f} | RMSE (train): {rmse_train_rf:.3f}')
print(f'R² (test) : {r2_test_rf:.3f}  | RMSE (test) : {rmse_test_rf:.3f}')
print('Note: May perform well, but is harder to explain than a simple line.')


## 7) Cross-Validation (Robustness Check)

A single train/test split can be lucky or unlucky, depending on which points happen to land in the test
set. Five fold cross-validation gives us a more robust picture: it splits the full dataset into five equal
folds, trains on four of them and tests on the fifth, and repeats this five times so that every fold is
used as the test set exactly once. Averaging the resulting scores tells us how each model typically
performs across different train/test splits, rather than relying on the one split we happened to draw
earlier.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)   # 5 folds, shuffled so the split isn't ordered by rainfall

# Score each model with 5-fold cross-validation on the *full* dataset (X, y), not just the earlier split
cv_lin  = cross_val_score(LinearRegression(), X, y, scoring='r2', cv=cv)
cv_p5   = cross_val_score(poly5,             X, y, scoring='r2', cv=cv)
cv_rf   = cross_val_score(rf,                X, y, scoring='r2', cv=cv)

print('=== 5-fold CV R² (mean ± sd) ===')
print(f'Linear     : {cv_lin.mean():.3f} ± {cv_lin.std():.3f}')
print(f'Poly deg=5 : {cv_p5.mean():.3f} ± {cv_p5.std():.3f}')
print(f'RandomForest: {cv_rf.mean():.3f} ± {cv_rf.std():.3f}')


## 8) Interpretation: Occam's Razor

- The **linear model** is simple, and it explains most of the variation in nitrate with a single, clearly
  communicable equation.
- The **polynomial model** often looks impressive on the training data, but tends to generalise worse, and
  that gap between train and test performance is exactly the overfitting we set out to detect.
- The **random forest** may perform competitively, but it is a black box, and it is harder to justify to
  decision makers who need to understand why the model predicts what it does.

**Decision-maker summary (what you would put in a report):**

Nitrate increases by about m mg/L for every 1 mm increase in rainfall, based on the linear model, with an
R² of around the value reported above on test data. A simple linear model provides a clear explanation
and adequate predictive capability, and adding complexity did not produce a meaningful improvement on the
test set. In line with Occam's Razor, we recommend the linear model for policy and reporting purposes.
